In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
import numpy as np
import torch

In [4]:
torch.cuda.is_available()

True

In [5]:
df = pd.read_csv('train.csv')
print(df.shape)
print(df.describe(include = 'str'))
df.head()

(36473, 5)
                      id                       anchor       target context
count              36473                        36473        36473   36473
unique             36473                          733        29340     106
top     37d61fd2272659b1  component composite coating  composition     H01
freq                   1                          152           24    2186


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


In [6]:
df['input'] = 'TEXT1: ' + df.context + '; TEXT2: ' + df.target + '; ANC1: ' + df.anchor
df.head()

,id,anchor,target,context,score,input
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50,TEXT1: A47; TEXT2: abatement of pollution; ANC...
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75,TEXT1: A47; TEXT2: act of abating; ANC1: abate...
2,36d72442aefd8232,abatement,active catalyst,A47,0.25,TEXT1: A47; TEXT2: active catalyst; ANC1: abat...
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50,TEXT1: A47; TEXT2: eliminating process; ANC1: ...
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00,TEXT1: A47; TEXT2: forest region; ANC1: abatement


In [7]:
df.iloc[0]['input']

'TEXT1: A47; TEXT2: abatement of pollution; ANC1: abatement'

In [8]:
ds = Dataset.from_pandas(df)
ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'score', 'input'],
    num_rows: 36473
})

In [9]:
model_name = 'microsoft/deberta-v3-small'

In [10]:
tokz = AutoTokenizer.from_pretrained(model_name)

In [11]:
tokz.tokenize(ds[0]['input'])

['▁TEXT',
 '1',
 ':',
 '▁A',
 '47',
 ';',
 '▁TEXT',
 '2',
 ':',
 '▁abatement',
 '▁of',
 '▁pollution',
 ';',
 '▁ANC',
 '1',
 ':',
 '▁abatement']

In [12]:
def tok_func(x):
    return tokz(x['input'])

In [13]:
tok_ds = ds.map(tok_func, batched = True)

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

In [14]:
row = tok_ds[0]
print(row['input'])
for i1, i2 in zip(tokz.tokenize(row['input']), row['input_ids']):
    print(f'"{i1}": "{i2}"')

TEXT1: A47; TEXT2: abatement of pollution; ANC1: abatement
"▁TEXT": "54453"
"1": "435"
":": "294"
"▁A": "336"
"47": "5753"
";": "346"
"▁TEXT": "54453"
"2": "445"
":": "294"
"▁abatement": "47284"
"▁of": "265"
"▁pollution": "6435"
";": "346"
"▁ANC": "23702"
"1": "435"
":": "294"
"▁abatement": "47284"


In [15]:
tokz.vocab['2']

445

In [16]:
tok_ds = tok_ds.rename_columns({'score': 'labels'})

In [17]:
dds = tok_ds.train_test_split(0.25, seed=42)
dds

DatasetDict({
    train: Dataset({
        features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 27354
    })
    test: Dataset({
        features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9119
    })
})

In [18]:
bs = 128
epochs = 4
lr = 8e-5

In [20]:
args = TrainingArguments(
    'outputs',
    learning_rate = lr,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'cosine',
    fp16 = True,
    eval_strategy = 'epoch',
    per_device_train_batch_size = bs,
    per_device_eval_batch_size = bs*2,
    num_train_epochs = epochs,
    weight_decay = 0.01,
    report_to = 'none',
    dataloader_pin_memory=True
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
def corr(x, y):
    return np.corrcoef(x, y)[0][1]
def corr_d(eval_pred):
    return {'pearson': corr(*eval_pred)}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 1, dtype=torch.float32)
trainer = Trainer(model, 
                  args, 
                  train_dataset = dds['train'], 
                  eval_dataset = dds['test'],
                  processing_class = tokz,
                  compute_metrics = corr_d
                )

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias         

In [25]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Pearson
1,No log,0.043020,0.772373
2,No log,0.026102,0.803941
3,0.035974,0.024563,0.815736
4,0.035974,0.023820,0.819267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=856, training_loss=0.028173457796328537, metrics={'train_runtime': 4427.4902, 'train_samples_per_second': 24.713, 'train_steps_per_second': 0.193, 'total_flos': 658937160751860.0, 'train_loss': 0.028173457796328537, 'epoch': 4.0})

In [26]:
eval_df = pd.read_csv('test.csv')
eval_df.describe()

,id,anchor,target,context
count,36,36,36,36
unique,36,34,36,29
top,4112d61851461f60,el display,inorganic photoconductor drum,G02
freq,1,2,1,3


In [27]:
eval_df['input'] = 'TEXT1: ' + eval_df.context + '; TEXT2: ' + eval_df.target + '; ANC1: ' + eval_df.anchor
eval_ds = Dataset.from_pandas(eval_df).map(tok_func, batched=True)

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [28]:
preds = trainer.predict(eval_ds).predictions.astype(float)
preds

array([[ 0.52294922],
       [ 0.65478516],
       [ 0.54394531],
       [ 0.36254883],
       [ 0.06384277],
       [ 0.50195312],
       [ 0.43652344],
       [ 0.01395416],
       [ 0.24694824],
       [ 1.08691406],
       [ 0.25415039],
       [ 0.29614258],
       [ 0.74951172],
       [ 0.80419922],
       [ 0.75683594],
       [ 0.52734375],
       [ 0.25219727],
       [-0.01548767],
       [ 0.66796875],
       [ 0.34619141],
       [ 0.40966797],
       [ 0.28466797],
       [ 0.0198822 ],
       [ 0.23083496],
       [ 0.58789062],
       [-0.04101562],
       [-0.06201172],
       [-0.04034424],
       [-0.04208374],
       [ 0.71289062],
       [ 0.33544922],
       [ 0.03320312],
       [ 0.68164062],
       [ 0.4831543 ],
       [ 0.4621582 ],
       [ 0.15393066]])

In [29]:
preds = np.clip(preds, 0, 1)
preds

array([[0.52294922],
       [0.65478516],
       [0.54394531],
       [0.36254883],
       [0.06384277],
       [0.50195312],
       [0.43652344],
       [0.01395416],
       [0.24694824],
       [1.        ],
       [0.25415039],
       [0.29614258],
       [0.74951172],
       [0.80419922],
       [0.75683594],
       [0.52734375],
       [0.25219727],
       [0.        ],
       [0.66796875],
       [0.34619141],
       [0.40966797],
       [0.28466797],
       [0.0198822 ],
       [0.23083496],
       [0.58789062],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.71289062],
       [0.33544922],
       [0.03320312],
       [0.68164062],
       [0.4831543 ],
       [0.4621582 ],
       [0.15393066]])

In [30]:
import datasets

submission = datasets.Dataset.from_dict({
    'id': eval_ds['id'],
    'score': preds
})

submission.to_csv('submission.csv', index=False)

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1075